# Limpeza de `uso_mensal.csv`

O diagnóstico (notebook 10) não achou linha a remover. A limpeza deixa o arquivo pronto para análise e resolve os meses
que faltam, com as regras de 07/11/2026 no `decisoes.md`:

1. Em 2025 só existe linha no mês com treino: usuário-mês sem linha = 0 treino, 0 minuto, 0 dia ativo; `sessoes_app` desconhecida.
2. 2025 e 2026 se comparam só pelas linhas com treino (ou com 2025 preenchido pela regra 1).
3. Em 2026, mês sem linha fica desconhecido (os treinos de 2026 virão de `atividades.parquet`, na limpeza dela).
4. `minutos` deste arquivo vale como aproximação só em 2025.
5. No modelo, só meses fechados antes da data da previsão: por isso a tabela traz o uso do mês anterior.

| Tabela | Uma linha por | Liga com |
|---|---|---|
| `uso_mensal_limpo.parquet` | usuário × mês, como no arquivo | `user_id` + `mes` |
| `uso_mes.parquet` | assinatura × mês, as mesmas linhas do painel | `assinaturas_mes.parquet` por `assinatura_id` + `mes` |

**Zero não é desconhecido:** `origem_valor` diz de onde veio cada linha, e o desconhecido fica vazio (`<NA>`), nunca 0.
Nada em `dados/brutos/` é alterado: o hash é conferido no início e no fim.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
INICIO_ARQUIVO = pd.Timestamp("2025-01-01")
INICIO_2026 = pd.Timestamp("2026-01-01")

ARQUIVOS = ["uso_mensal.csv", "usuarios.csv"]
NUM = ["atividades", "atividades_ar_livre", "atividades_video", "minutos", "dias_ativos", "sessoes_app"]
TREINO = ["atividades", "atividades_ar_livre", "atividades_video", "minutos", "dias_ativos"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(tabela, passo, n, obs=""):
    registro.append({"tabela": tabela, "passo": passo, "linhas": n, "observacao": obs})
    print(f"[{tabela}] {passo}: {n:,} linhas" + (f" ({obs})" if obs else ""))

## 0. Leitura e as checagens do diagnóstico que garantem a limpeza

Se alguma voltar a falhar, a limpeza para: as regras abaixo supõem estes zeros.

In [2]:
um = pd.read_csv(BRUTOS / "uso_mensal.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False, usecols=["user_id", "email"])
assin = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")
trials = pd.read_parquet(PROCESSADOS / "trials.parquet")
painel = pd.read_parquet(PROCESSADOS / "assinaturas_mes.parquet", columns=["assinatura_id", "user_id", "mes"])

assert len(um) == 591_662
email = usuarios.set_index("user_id")["email"].str.strip()
teste = email.str.lower().str.endswith("@vitaliza.com.br")
duplicada = email.str.contains(r"[A-Z]") & email.str.lower().duplicated(keep=False)

for c in NUM:
    assert um[c].str.fullmatch(r"\d+").all(), c
um[NUM] = um[NUM].astype("int64")
um["mes"] = pd.to_datetime(um["mes"], format="%Y-%m")

assert not um["user_id"].map(teste).any() and not um["user_id"].map(duplicada).any()
assert not um.duplicated(["user_id", "mes"]).any()
assert (um["atividades_ar_livre"] + um["atividades_video"] == um["atividades"]).all()
assert ((um["atividades"] > 0) == (um["minutos"] > 0)).all() and ((um["atividades"] > 0) == (um["dias_ativos"] > 0)).all()
assert (um.loc[um["mes"] < INICIO_2026, "atividades"] > 0).all()   # regra 1: 2025 só tem mês com treino
registrar("uso_mensal_limpo", "0. Arquivo bruto", len(um), "0 de teste, 0 duplicadas, 0 repetidas: nada a remover")

[uso_mensal_limpo] 0. Arquivo bruto: 591,662 linhas (0 de teste, 0 duplicadas, 0 repetidas: nada a remover)


## 1. Tabela: `uso_mensal_limpo.parquet`

O arquivo com números, `mes` como data, a marca do mês sem uso e a situação da conta no mês.

In [3]:
d0, d1 = um["mes"], um["mes"] + pd.offsets.MonthBegin(1)
a = assin.set_index("user_id")
t = trials.set_index("user_id")
ini = um["user_id"].map(a["inicio"])
fim = um["user_id"].map(a["data_fim"]).fillna(pd.Timestamp("2099-01-01"))
t_ini = um["user_id"].map(t["trial_inicio"])
t_fim = um["user_id"].map(t["data_desfecho"]).fillna(pd.Timestamp("2099-01-01"))
um["situacao"] = np.select(
    [ini.notna() & (ini < d1) & (fim >= d0), t_ini.notna() & (t_ini < d1) & (t_fim >= d0), ini.notna() & (fim < d0)],
    ["assinante", "em_teste", "ex_assinante"], "nunca_assinou")
um["mes_sem_uso"] = (um["atividades"] == 0) & (um["sessoes_app"] == 0)
um["com_treino"] = um["atividades"] > 0

limpo = um[["user_id", "mes", "situacao"] + NUM + ["com_treino", "mes_sem_uso"]].sort_values(["user_id", "mes"]).reset_index(drop=True)
display(pd.crosstab(limpo["mes"].dt.year.rename("ano"), limpo["situacao"], margins=True, margins_name="total"))
print(f"meses sem uso (sem treino e sem abrir o app): {limpo['mes_sem_uso'].sum():,}, "
      f"todos de 2026: {limpo.loc[limpo['mes_sem_uso'], 'mes'].min() >= INICIO_2026}")
registrar("uso_mensal_limpo", "1. Tipos, situação e marcas", len(limpo), f"{limpo['mes_sem_uso'].sum():,} meses sem uso")

situacao,assinante,em_teste,ex_assinante,nunca_assinou,total
ano,,,,,
2025,123465,8868,1060,18837,152230
2026,132169,10733,102049,194481,439432
total,255634,19601,103109,213318,591662


meses sem uso (sem treino e sem abrir o app): 41,493, todos de 2026: True
[uso_mensal_limpo] 1. Tipos, situação e marcas: 591,662 linhas (41,493 meses sem uso)


## 2. Uso de uma conta num mês, com as regras de preenchimento

`uso_em(chaves)` devolve o uso de cada par conta × mês e de onde ele veio:

| `origem_valor` | Quando | Treinos, minutos, dias | `sessoes_app` |
|---|---|---|---|
| `arquivo` | há linha no arquivo | do arquivo | do arquivo |
| `zero_2025` | 2025, sem linha (regra 1) | 0 | desconhecida |
| `sem_linha_2026` | 2026, sem linha (regra 3) | desconhecido | desconhecida |
| `antes_do_arquivo` | antes de jan/2025 | desconhecido | desconhecida |

In [4]:
def uso_em(chaves):
    """chaves: DataFrame com user_id e mes (1º dia do mês). Devolve as colunas de uso e origem_valor, no mesmo índice."""
    r = chaves[["user_id", "mes"]].merge(limpo[["user_id", "mes"] + NUM], on=["user_id", "mes"], how="left",
                                         validate="many_to_one")
    r.index = chaves.index
    achou = r["atividades"].notna()
    origem = np.select([achou, chaves["mes"] < INICIO_ARQUIVO, chaves["mes"] < INICIO_2026],
                       ["arquivo", "antes_do_arquivo", "zero_2025"], "sem_linha_2026")
    out = r[NUM].astype("Int64")
    zero = origem == "zero_2025"
    out.loc[zero, TREINO] = 0
    out["origem_valor"] = origem
    return out

## 3. Tabela: `uso_mes.parquet`

As linhas do painel, com o uso do mês e o do mês anterior (`_mes_anterior`). O mês anterior vem da mesma conta, mesmo
quando ela ainda estava em teste ou não pagava.

In [5]:
base = painel.copy()
uso = uso_em(base)
anterior = uso_em(base.assign(mes=base["mes"] - pd.offsets.MonthBegin(1)))
anterior.columns = [f"{c}_mes_anterior" for c in anterior.columns]
um_mes = pd.concat([base, uso, anterior], axis=1)

contagem = um_mes["origem_valor"].value_counts()
display(pd.DataFrame({"linhas": contagem, "esperado": pd.Series(
    {"arquivo": 255_634, "antes_do_arquivo": 45_198, "zero_2025": 23_249, "sem_linha_2026": 3_690})}))
assert (contagem == pd.Series({"arquivo": 255_634, "antes_do_arquivo": 45_198, "zero_2025": 23_249,
                               "sem_linha_2026": 3_690})).all()
display(pd.crosstab(um_mes["origem_valor"], um_mes["origem_valor_mes_anterior"], margins=True, margins_name="total"))
registrar("uso_mes", "3. Uma linha por linha do painel", len(um_mes),
          ", ".join(f"{k}: {v:,}" for k, v in contagem.items()))

,linhas,esperado
arquivo,255634,255634
antes_do_arquivo,45198,45198
zero_2025,23249,23249
sem_linha_2026,3690,3690


origem_valor_mes_anterior,antes_do_arquivo,arquivo,sem_linha_2026,zero_2025,total
origem_valor,,,,,
antes_do_arquivo,45198,0,0,0,45198
arquivo,8765,222125,8744,16000,255634
sem_linha_2026,0,3132,334,224,3690
zero_2025,1198,8775,0,13276,23249
total,55161,234032,9078,29500,327771


[uso_mes] 3. Uma linha por linha do painel: 327,771 linhas (arquivo: 255,634, antes_do_arquivo: 45,198, zero_2025: 23,249, sem_linha_2026: 3,690)


## 4. Conferências

In [6]:
do_arquivo = um_mes.loc[um_mes["origem_valor"] == "arquivo"]
chave = pd.MultiIndex.from_frame(do_arquivo[["user_id", "mes"]])
ref = limpo.set_index(["user_id", "mes"]).loc[chave, NUM]

conf = {
    "linhas do arquivo: valores iguais aos do arquivo": (do_arquivo[NUM].astype("int64").values == ref.values).all(),
    "zero_2025: treinos, minutos e dias = 0": (um_mes.loc[um_mes["origem_valor"] == "zero_2025", TREINO] == 0).all().all(),
    "zero_2025: sessoes_app desconhecida": um_mes.loc[um_mes["origem_valor"] == "zero_2025", "sessoes_app"].isna().all(),
    "sem_linha_2026 e antes_do_arquivo: tudo desconhecido":
        um_mes.loc[um_mes["origem_valor"].isin(["sem_linha_2026", "antes_do_arquivo"]), NUM].isna().all().all(),
    "nenhum zero fora de 'arquivo' e 'zero_2025'":
        not (um_mes.loc[~um_mes["origem_valor"].isin(["arquivo", "zero_2025"]), NUM] == 0).any().any(),
}

# o uso do mês anterior é o da linha do mês anterior, quando a assinatura também estava no painel nele
prev = um_mes[["assinatura_id", "mes"] + NUM].assign(mes=lambda x: x["mes"] + pd.offsets.MonthBegin(1))
par = um_mes.merge(prev, on=["assinatura_id", "mes"], suffixes=("", "_linha_anterior"))
iguais = all(par[f"{c}_mes_anterior"].astype("float").fillna(-1).equals(par[f"{c}_linha_anterior"].astype("float").fillna(-1))
             for c in NUM)
conf[f"mês anterior = linha do mês anterior ({len(par):,} pares)"] = iguais
tabela = pd.Series(conf, name="ok").to_frame()
display(tabela)
assert tabela["ok"].all()

tot = pd.DataFrame({
    "arquivo (linhas que estão no painel)": ref[["atividades", "minutos"]].sum(),
    "uso_mes (origem arquivo)": do_arquivo[["atividades", "minutos"]].astype("int64").sum(),
})
display(tot)
assert (tot.iloc[:, 0] == tot.iloc[:, 1]).all()

,ok
linhas do arquivo: valores iguais aos do arquivo,True
"zero_2025: treinos, minutos e dias = 0",True
zero_2025: sessoes_app desconhecida,True
sem_linha_2026 e antes_do_arquivo: tudo desconhecido,True
nenhum zero fora de 'arquivo' e 'zero_2025',True
"mês anterior = linha do mês anterior (295,188 pares)",True


,arquivo (linhas que estão no painel),uso_mes (origem arquivo)
atividades,1727772,1727772
minutos,76551900,76551900


### 4.1 Uso médio dos assinantes por mês

Só linhas com valor (arquivo ou zero de 2025). Treinos aqui são só os registrados no app, nos dois anos.

In [7]:
display(um_mes.loc[um_mes["origem_valor"].isin(["arquivo", "zero_2025"])]
        .groupby("mes")[["atividades", "dias_ativos", "minutos", "sessoes_app"]].mean().astype(float).round(2).T)

mes,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01,2026-09-01
atividades,7.16,5.34,6.04,6.42,6.59,5.76,5.81,6.34,6.43,6.75,6.46,5.58,7.11,5.35,6.03,6.26,6.43,5.72,5.70,6.35,6.56
dias_ativos,5.83,4.50,5.07,5.32,5.42,4.84,4.88,5.25,5.29,5.56,5.32,4.74,5.79,4.51,5.06,5.18,5.34,4.81,4.81,5.26,5.41
minutos,317.35,235.94,267.06,284.06,291.59,254.81,256.24,280.10,283.60,299.33,286.26,246.93,315.70,237.54,268.10,278.63,285.94,254.08,252.44,280.75,290.77
sessoes_app,19.87,15.96,17.52,18.59,19.08,17.14,17.42,18.69,18.90,19.54,18.80,16.81,17.62,13.73,8.41,8.70,8.91,8.04,8.03,8.81,9.07


### 4.2 `sessoes_app` muda de patamar em março de 2026

Os treinos por assinante ficam estáveis, mas as aberturas do app caem pela metade em mar/2026. Se fosse comportamento
(ex.: reação ao reajuste do mensal em 01/03/2026), as contas grátis não seriam afetadas. A tabela mede aberturas por treino,
só nos meses com treino, por situação da conta e por sistema.

In [8]:
ct = limpo.loc[limpo["com_treino"]].assign(sessoes_por_treino=lambda x: x["sessoes_app"] / x["atividades"],
                                           mes_txt=lambda x: x["mes"].dt.strftime("%Y-%m"))
por_sit = ct.groupby(["mes_txt", "situacao"])["sessoes_por_treino"].mean().unstack().round(2)
display(por_sit.loc["2025-10":])
antes = ct.loc[ct["mes_txt"].between("2025-10", "2026-02"), "sessoes_por_treino"]
depois = ct.loc[ct["mes_txt"].between("2026-03", "2026-09"), "sessoes_por_treino"]
queda = por_sit.loc["2026-03"] / por_sit.loc["2026-02"] - 1
print(f"aberturas por treino: {antes.mean():.2f} (out/25 a fev/26) -> {depois.mean():.2f} (mar a set/26); "
      f"queda de fev para mar/26 por situação: {queda.round(2).to_dict()}")
print(f"meses com menos de 2 aberturas por treino: fev/26 {(ct.loc[ct['mes_txt'] == '2026-02', 'sessoes_por_treino'] < 2).mean():.1%}, "
      f"mar/26 {(ct.loc[ct['mes_txt'] == '2026-03', 'sessoes_por_treino'] < 2).mean():.1%}")

situacao,assinante,em_teste,ex_assinante,nunca_assinou
mes_txt,,,,
2025-10,2.78,3.21,3.70,3.50
2025-11,2.79,3.29,3.93,3.57
2025-12,2.85,3.38,3.93,3.61
2026-01,2.78,3.24,3.94,3.51
2026-02,2.85,3.34,3.80,3.45
2026-03,1.58,1.87,2.12,2.02
2026-04,1.57,1.82,2.15,2.00
2026-05,1.55,1.80,2.10,2.04
2026-06,1.59,1.85,2.18,2.02


aberturas por treino: 2.93 (out/25 a fev/26) -> 1.64 (mar a set/26); queda de fev para mar/26 por situação: {'assinante': -0.45, 'em_teste': -0.44, 'ex_assinante': -0.44, 'nunca_assinou': -0.41}
meses com menos de 2 aberturas por treino: fev/26 0.0%, mar/26 69.2%


A queda é de um mês para o outro e do mesmo tamanho (cerca de 44%) em todas as situações, inclusive nas contas que nunca
assinaram, que o reajuste não atinge. Isso aponta para mudança na forma de contar as aberturas, não para mudança de
comportamento. Fica como hipótese a confirmar com o dono do dado.

## 5. Saídas e integridade dos brutos

In [9]:
saidas = {"uso_mensal_limpo.parquet": limpo, "uso_mes.parquet": um_mes.reset_index(drop=True)}
for nome, df in saidas.items():
    df.to_parquet(PROCESSADOS / nome, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(PROCESSADOS / nome), df)
    print(f"dados/processados/{nome}: {len(df):,} linhas x {df.shape[1]} colunas")

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_uso_mensal_registro.csv", index=False)
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/processados/uso_mensal_limpo.parquet: 591,662 linhas x 11 colunas


dados/processados/uso_mes.parquet: 327,771 linhas x 17 colunas


,tabela,passo,linhas,observacao
0,uso_mensal_limpo,0. Arquivo bruto,591662,"0 de teste, 0 duplicadas, 0 repetidas: nada a ..."
1,uso_mensal_limpo,"1. Tipos, situação e marcas",591662,"41,493 meses sem uso"
2,uso_mes,3. Uma linha por linha do painel,327771,"arquivo: 255,634, antes_do_arquivo: 45,198, ze..."


dados/brutos/ inalterado: True
